# OpenAI Agents: 交接、围栏、追溯

OpenAI Agents 的SDK 是基于 Responses API 构建的轻量多agent框架。五个原语：Agent、Handoff、Guardrail、Session、Tracing。Handoffs是名为`transfer_to<agent>`的工具。Guardrail 在输入和输出拦截。Tracing默认开启。

## 问题描述

不能干净代理的Agents最后会把所有东西都塞到一个prompt里。没有guardrails的agents会吐出PII、违反策略的输出，或者一直循环到死。OpenAI 的SDK 把让多个agent工作变得可控的三个原语固定下来。

## 基本概念

### 五个原语

1. **Agent**。 LLM+指令+工具+交接。
2. **Handoff**。 代理给另外一个agent。
3. **Guardrail**。 输入校验（第一个agent），输出校验（以后一个agent），或者工具调用校验（每个函数工具）
4. **Session**。 跨轮的自动会话历史。
5. **Tracing**。 LLM生成、工具调用、handoffs、guardrails内置的钩子。

### Handoffs 作为工具

模型在工具中能看到`transfer_to_biling_agent`，调用它向运行时发送信号：
1. 拷贝会话上下文（或者压缩）
2. 通过目标agent的指令初始化
3. 目标agent继续执行

这就是监管者模式的产品化。

### Guardrails

三道门槛：
1. 输入围栏。 在第一个agent输入时执行，在任何LLM调用前拒绝不安全和越界的请求。
2. 输出围栏。 在最后一个agent输出时执行。捕捉PII泄漏、政策违反、错误格式的回复。
3. 工具围栏。  在每个函数工具调用。校验参数、权限、审计执行。

模式：
- 并行。   围栏LLM的主LLM同时跑，尾延迟低。如果围栏触发，主LLM的工作被丢弃，会导致token浪费。
- 阻塞。  围栏LLM先跑，如果触发，主LLM调用没有token浪费。

触发时会抛出`InputGuardrailTripwireTriggered/OutputGuardrailTripwireTriggered`。

### 什么时候失败
- 交接漂移。  A 交接给B，B又交接回A。加上一个跳数计数器。
- 绕过围栏。  工具围栏只对函数工具有用，但是内置工具（文件读取器、网页拉取）需要单独的策略。
- 过渡监视。  Span里有敏感内容。

# 开始编码

对应本章核心：**Handoff = `transfer_to_*` 工具**、**输入/输出/工具三道 Guardrail**、**Session 跨轮历史**、**Tracing（Trace + Span + 钩子）**、**交接跳数防漂移**。  
先用玩具运行时跑通交接与围栏触发；再用 **LangChain + DeepSeek** 挂真实多 agent（不硬凑 PyTorch）。


## 1. 教学玩具：五原语迷你运行时

- **Agent**：instructions + tools + handoffs。
- **Handoff**：名为 `transfer_to_<agent>` 的特殊工具。
- **Guardrail**：input / output / tool；阻塞模式先检再跑；触发抛 Tripwire。
- **Session**：跨轮消息历史。
- **Tracing**：默认记录 span；钩子可订阅 `on_span_*`。


In [ ]:
from __future__ import annotations

import re
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from typing import Any, Callable, Literal

ToolFn = Callable[[dict[str, Any], "RunContext"], str]
GuardFn = Callable[[str, "RunContext"], tuple[bool, str]]
HookFn = Callable[["Span"], None]


class InputGuardrailTripwireTriggered(Exception):
    """输入围栏触发。"""


class OutputGuardrailTripwireTriggered(Exception):
    """输出围栏触发。"""


class ToolGuardrailTripwireTriggered(Exception):
    """工具围栏触发。"""


@dataclass
class Span:
    """Trace 内一段可计量工作。"""

    name: str
    kind: Literal["llm", "tool", "handoff", "guardrail", "agent"]
    start_ts: float
    end_ts: float | None = None
    input: Any = None
    output: Any = None
    error: str | None = None
    span_id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])

    def close(self, output: Any = None, error: str | None = None) -> None:
        self.end_ts = time.time()
        self.output = output
        self.error = error


@dataclass
class Trace:
    """一次完整运行的轨迹。"""

    trace_id: str = field(default_factory=lambda: uuid.uuid4().hex[:10])
    spans: list[Span] = field(default_factory=list)


class Tracer:
    """默认开启的追溯 + 钩子。"""

    def __init__(self) -> None:
        self.trace = Trace()
        self.hooks: dict[str, list[HookFn]] = {
            "on_span_start": [],
            "on_span_end": [],
        }

    def on(self, event: str, fn: HookFn) -> None:
        """
        Args:
            event: ``on_span_start`` / ``on_span_end``。
            fn: 钩子回调。
        """
        self.hooks.setdefault(event, []).append(fn)

    def start_span(self, name: str, kind: Span.__annotations__["kind"], input: Any = None) -> Span:
        span = Span(name=name, kind=kind, start_ts=time.time(), input=input)
        self.trace.spans.append(span)
        for fn in self.hooks.get("on_span_start", []):
            fn(span)
        return span

    def end_span(self, span: Span, output: Any = None, error: str | None = None) -> None:
        span.close(output=output, error=error)
        for fn in self.hooks.get("on_span_end", []):
            fn(span)


@dataclass
class Session:
    """跨轮自动会话历史。"""

    session_id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    messages: list[dict[str, str]] = field(default_factory=list)

    def add(self, role: str, content: str) -> None:
        self.messages.append({"role": role, "content": content})

    def render(self, limit: int = 12) -> str:
        return "\n".join(f"{m['role']}: {m['content']}" for m in self.messages[-limit:])


@dataclass
class Tool:
    name: str
    description: str
    fn: ToolFn
    guardrail: GuardFn | None = None  # 工具围栏：校验参数序列化文本


@dataclass
class Agent:
    name: str
    instructions: str
    tools: list[Tool] = field(default_factory=list)
    handoffs: list[str] = field(default_factory=list)  # 目标 agent 名
    # 玩具里用策略函数代替 LLM
    policy: Callable[[str, "RunContext", "Agent"], dict[str, Any]] | None = None

    def transfer_tools(self) -> list[str]:
        """Handoff 暴露给模型的工具名。"""
        return [f"transfer_to_{h}" for h in self.handoffs]


@dataclass
class RunContext:
    session: Session
    tracer: Tracer
    hop: int = 0
    max_hops: int = 4
    meta: dict[str, Any] = field(default_factory=dict)


@dataclass
class Guardrails:
    input: list[GuardFn] = field(default_factory=list)
    output: list[GuardFn] = field(default_factory=list)
    mode: Literal["blocking", "parallel"] = "blocking"


class AgentRuntime:
    """OpenAI Agents 风格迷你运行时。"""

    def __init__(
        self,
        agents: dict[str, Agent],
        *,
        entry: str,
        guardrails: Guardrails | None = None,
        session: Session | None = None,
        tracer: Tracer | None = None,
        max_hops: int = 4,
    ) -> None:
        self.agents = agents
        self.entry = entry
        self.guardrails = guardrails or Guardrails()
        self.session = session or Session()
        self.tracer = tracer or Tracer()
        self.max_hops = max_hops

    def _check_guards(self, kind: str, text: str, ctx: RunContext, guards: list[GuardFn]) -> None:
        span = self.tracer.start_span(f"guardrail:{kind}", "guardrail", input=text[:200])
        try:
            for g in guards:
                ok, reason = g(text, ctx)
                if not ok:
                    self.tracer.end_span(span, error=reason)
                    if kind == "input":
                        raise InputGuardrailTripwireTriggered(reason)
                    if kind == "output":
                        raise OutputGuardrailTripwireTriggered(reason)
                    raise ToolGuardrailTripwireTriggered(reason)
            self.tracer.end_span(span, output="pass")
        except Exception:
            if span.end_ts is None:
                self.tracer.end_span(span, error="raised")
            raise

    def _call_tool(self, agent: Agent, name: str, args: dict[str, Any], ctx: RunContext) -> str:
        # handoff 工具
        if name.startswith("transfer_to_"):
            target = name[len("transfer_to_") :]
            span = self.tracer.start_span(name, "handoff", input={"from": agent.name, "to": target})
            if ctx.hop >= ctx.max_hops:
                self.tracer.end_span(span, error="max hops")
                raise RuntimeError(f"handoff drift: max_hops={ctx.max_hops}")
            if target not in self.agents:
                self.tracer.end_span(span, error="unknown target")
                raise KeyError(target)
            ctx.hop += 1
            self.tracer.end_span(span, output={"hop": ctx.hop})
            return f"__HANDOFF__:{target}"

        tool = next((t for t in agent.tools if t.name == name), None)
        if tool is None:
            raise KeyError(f"unknown tool {name}")
        span = self.tracer.start_span(name, "tool", input=args)
        try:
            if tool.guardrail is not None:
                ok, reason = tool.guardrail(str(args), ctx)
                if not ok:
                    self.tracer.end_span(span, error=reason)
                    raise ToolGuardrailTripwireTriggered(reason)
            out = tool.fn(args, ctx)
            self.tracer.end_span(span, output=out[:200] if isinstance(out, str) else out)
            return out
        except ToolGuardrailTripwireTriggered:
            raise
        except Exception as e:
            self.tracer.end_span(span, error=str(e))
            raise

    def _run_agent_once(self, agent: Agent, user_text: str, ctx: RunContext) -> dict[str, Any]:
        span = self.tracer.start_span(f"agent:{agent.name}", "agent", input=user_text[:200])
        try:
            policy = agent.policy or default_policy
            # llm span（玩具：策略函数）
            llm_span = self.tracer.start_span(f"llm:{agent.name}", "llm", input=user_text[:200])
            decision = policy(user_text, ctx, agent)
            self.tracer.end_span(llm_span, output=decision)
            self.tracer.end_span(span, output=decision.get("type"))
            return decision
        except Exception as e:
            self.tracer.end_span(span, error=str(e))
            raise

    def run(self, user_text: str, *, is_first_turn: bool = True) -> str:
        """
        Args:
            user_text: 用户输入。
            is_first_turn: 是否跑输入围栏（首个 agent / 新用户消息）。

        Returns:
            final: 最终助手输出。
        """
        ctx = RunContext(session=self.session, tracer=self.tracer, max_hops=self.max_hops)
        self.session.add("user", user_text)

        if is_first_turn and self.guardrails.input:
            if self.guardrails.mode == "blocking":
                self._check_guards("input", user_text, ctx, self.guardrails.input)
            else:
                # 并行模式示意：围栏与「主工作准备」同时；触发则丢弃主工作
                with ThreadPoolExecutor(max_workers=2) as pool:
                    fut_g = pool.submit(self._check_guards, "input", user_text, ctx, self.guardrails.input)
                    fut_w = pool.submit(lambda: time.sleep(0.01) or "prepared")
                    for fut in as_completed([fut_g, fut_w]):
                        fut.result()  # 围栏失败会抛

        current = self.entry
        final = ""
        steps = 0
        while steps < 16:
            steps += 1
            agent = self.agents[current]
            decision = self._run_agent_once(agent, user_text, ctx)
            dtype = decision.get("type")
            if dtype == "tool":
                out = self._call_tool(agent, decision["name"], decision.get("args") or {}, ctx)
                if out.startswith("__HANDOFF__:"):
                    current = out.split(":", 1)[1]
                    # 拷贝/压缩会话上下文：玩具里直接沿用 session
                    continue
                self.session.add("tool", f"{decision['name']}->{out}")
                user_text = f"Tool result: {out}\nOriginal: {user_text}"
                continue
            if dtype == "message":
                final = str(decision.get("content", ""))
                break
            raise RuntimeError(f"bad decision: {decision}")

        if self.guardrails.output:
            self._check_guards("output", final, ctx, self.guardrails.output)
        self.session.add("assistant", final)
        return final


def default_policy(user_text: str, ctx: RunContext, agent: Agent) -> dict[str, Any]:
    """极简路由：看关键词选 handoff / tool / message。"""
    # 已有工具结果 → 直接回复，避免工具死循环
    if user_text.startswith("Tool result:"):
        return {"type": "message", "content": f"[{agent.name}] {user_text.split(chr(10), 1)[0]}"}
    text = user_text.lower()
    billing_intent = (
        "bill" in text
        or "refund" in text
        or "发票" in user_text
        or "账单" in user_text
        or "退款" in user_text
    )
    if billing_intent and "billing" in agent.handoffs:
        return {"type": "tool", "name": "transfer_to_billing", "args": {}}
    if billing_intent and any(t.name == "lookup_order" for t in agent.tools):
        m = re.search(r"\d{3,}", user_text)
        return {"type": "tool", "name": "lookup_order", "args": {"order_id": m.group(0) if m else "0"}}
    return {"type": "message", "content": f"[{agent.name}] {agent.instructions[:40]} :: {user_text}"}


def guard_no_pii_input(text: str, ctx: RunContext) -> tuple[bool, str]:
    if re.search(r"\b\d{16,19}\b", text):
        return False, "input contains card-like number"
    return True, "ok"


def guard_no_pii_output(text: str, ctx: RunContext) -> tuple[bool, str]:
    if re.search(r"\b\d{16,19}\b", text):
        return False, "output leaks card-like number"
    return True, "ok"


def guard_tool_order_id(args_text: str, ctx: RunContext) -> tuple[bool, str]:
    if "'order_id': '0'" in args_text or '"order_id": "0"' in args_text or "order_id': 0" in args_text:
        return False, "order_id required"
    return True, "ok"


print("OpenAI Agents toy ready | handoff + guardrail + session + tracing")


## 2. 玩具示例：交接、围栏、跳数、Span 钩子


In [ ]:
def demo_openai_agents_toy() -> None:
    """断言 handoff、三道围栏、session、tracing、max_hops。"""

    def lookup_order(args: dict[str, Any], ctx: RunContext) -> str:
        return f"order={args.get('order_id')} status=paid"

    triage = Agent(
        name="triage",
        instructions="Route billing questions to billing.",
        handoffs=["billing"],
        policy=default_policy,
    )
    billing = Agent(
        name="billing",
        instructions="Handle invoices and refunds.",
        tools=[
            Tool("lookup_order", "lookup order", lookup_order, guardrail=guard_tool_order_id),
        ],
        handoffs=["triage"],  # 可能漂移
        policy=default_policy,
    )

    # 钩子：收集 span 名
    seen: list[str] = []
    tracer = Tracer()
    tracer.on("on_span_end", lambda s: seen.append(f"{s.kind}:{s.name}"))

    rt = AgentRuntime(
        {"triage": triage, "billing": billing},
        entry="triage",
        guardrails=Guardrails(input=[guard_no_pii_input], output=[guard_no_pii_output], mode="blocking"),
        session=Session(),
        tracer=tracer,
        max_hops=4,
    )

    # 1) 输入围栏
    try:
        rt.run("charge my card 4111111111111111")
        raise AssertionError("expected input tripwire")
    except InputGuardrailTripwireTriggered as e:
        print("input tripwire:", e)

    # 2) handoff triage → billing，再查单
    out = rt.run("我想退款，订单 12345")
    kinds = {s.kind for s in rt.tracer.trace.spans}
    assert "handoff" in kinds and "tool" in kinds and "guardrail" in kinds and "llm" in kinds
    assert any("order=12345" in (s.output or "") for s in rt.tracer.trace.spans if s.kind == "tool")
    assert "[billing]" in out or "paid" in out or "order" in out.lower()
    assert any(k.startswith("handoff:") for k in seen)
    print("handoff+tool ok; out=", out[:80], "; spans=", len(rt.tracer.trace.spans))

    # session 跨轮
    assert any(m["role"] == "user" for m in rt.session.messages)
    assert any(m["role"] == "assistant" for m in rt.session.messages)
    print("session turns:", len(rt.session.messages))

    # 3) 工具围栏：缺 order_id
    def force_bad_tool(user_text: str, ctx: RunContext, agent: Agent) -> dict[str, Any]:
        return {"type": "tool", "name": "lookup_order", "args": {"order_id": "0"}}

    billing.policy = force_bad_tool
    rt2 = AgentRuntime(
        {"triage": triage, "billing": billing},
        entry="billing",
        guardrails=Guardrails(),
        max_hops=2,
    )
    try:
        rt2.run("refund")
        raise AssertionError("expected tool tripwire")
    except ToolGuardrailTripwireTriggered as e:
        print("tool tripwire:", e)

    # 4) 交接漂移：A↔B 超过 max_hops
    def always_to_billing(user_text: str, ctx: RunContext, agent: Agent) -> dict[str, Any]:
        return {"type": "tool", "name": "transfer_to_billing", "args": {}}

    def always_to_triage(user_text: str, ctx: RunContext, agent: Agent) -> dict[str, Any]:
        return {"type": "tool", "name": "transfer_to_triage", "args": {}}

    a = Agent("triage", "t", handoffs=["billing"], policy=always_to_billing)
    b = Agent("billing", "b", handoffs=["triage"], policy=always_to_triage)
    rt3 = AgentRuntime({"triage": a, "billing": b}, entry="triage", max_hops=3)
    try:
        rt3.run("loop")
        raise AssertionError("expected hop limit")
    except RuntimeError as e:
        assert "max_hops" in str(e)
        print("handoff drift blocked:", e)

    # 5) 输出围栏
    def leak(user_text: str, ctx: RunContext, agent: Agent) -> dict[str, Any]:
        return {"type": "message", "content": "card 4111111111111111"}

    leaky = Agent("triage", "x", policy=leak)
    rt4 = AgentRuntime(
        {"triage": leaky},
        entry="triage",
        guardrails=Guardrails(output=[guard_no_pii_output]),
    )
    try:
        rt4.run("hi")
        raise AssertionError("expected output tripwire")
    except OutputGuardrailTripwireTriggered as e:
        print("output tripwire:", e)

    print("TOY DEMO OK")


demo_openai_agents_toy()


## 3. 生产级：LangChain + DeepSeek 多 Agent 交接

Triage / Billing 两个 agent；Handoff 仍是 `transfer_to_*` 工具。  
输入/输出围栏 + Tracing hooks；控制面工具：`run_agents` / `get_trace` / `get_session`。需 ``DEEPSEEK_API_KEY``。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
PROD_SESSION = Session()
PROD_TRACER = Tracer()


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def llm_policy_factory(system: str) -> Callable[[str, RunContext, Agent], dict[str, Any]]:
    """
    用 LLM 产出决策 JSON：message | tool(handoff/function)。

    Args:
        system: agent 指令。

    Returns:
        policy: Agent.policy。
    """

    def policy(user_text: str, ctx: RunContext, agent: Agent) -> dict[str, Any]:
        tool_names = [t.name for t in agent.tools] + agent.transfer_tools()
        prompt = (
            f"{system}\nSession:\n{ctx.session.render()}\n"
            f"Available tools: {tool_names}\n"
            f"User: {user_text}\n"
            "Reply ONLY JSON with one of:\n"
            '{"type":"message","content":"..."}\n'
            '{"type":"tool","name":"transfer_to_billing"|"lookup_order", "args":{...}}\n'
            "Chinese content. Keep short."
        )
        raw = str(get_llm().invoke(prompt).content).strip()
        m = re.search(r"\{[\s\S]*\}", raw)
        try:
            data = json.loads(m.group(0) if m else raw)
        except json.JSONDecodeError:
            return {"type": "message", "content": raw}
        if data.get("type") == "tool" and data.get("name") in tool_names:
            return {"type": "tool", "name": data["name"], "args": data.get("args") or {}}
        return {"type": "message", "content": str(data.get("content") or raw)}

    return policy


def build_prod_runtime() -> AgentRuntime:
    """
    Returns:
        runtime: triage + billing，带围栏与追溯。
    """
    global PROD_SESSION, PROD_TRACER
    PROD_SESSION = Session()
    PROD_TRACER = Tracer()

    def lookup_order(args: dict[str, Any], ctx: RunContext) -> str:
        oid = str(args.get("order_id", ""))
        return json.dumps({"order_id": oid, "status": "paid", "amount": 99}, ensure_ascii=False)

    triage = Agent(
        name="triage",
        instructions="你是分流助手。账单/发票/退款请 transfer_to_billing；其它直接简短回答。",
        handoffs=["billing"],
        policy=llm_policy_factory("Triage agent. Prefer handoff for billing topics."),
    )
    billing = Agent(
        name="billing",
        instructions="你是账单助手。需要订单号时调用 lookup_order，然后中文简短答复。",
        tools=[Tool("lookup_order", "Lookup order by id", lookup_order, guardrail=guard_tool_order_id)],
        handoffs=[],
        policy=llm_policy_factory("Billing agent. Use lookup_order when order id present."),
    )
    return AgentRuntime(
        {"triage": triage, "billing": billing},
        entry="triage",
        guardrails=Guardrails(
            input=[guard_no_pii_input],
            output=[guard_no_pii_output],
            mode="blocking",
        ),
        session=PROD_SESSION,
        tracer=PROD_TRACER,
        max_hops=4,
    )


PROD_RT = build_prod_runtime()


class RunArgs(BaseModel):
    text: str = Field(description="User message")


class EmptyArgs(BaseModel):
    pass


def run_agents_impl(text: str) -> str:
    """
    Returns:
        json: 最终回复或 tripwire。
    """
    global PROD_RT
    try:
        # 新会话工具调用时重建，避免污染；也可跨轮复用 session
        out = PROD_RT.run(text)
        return json.dumps({"status": "ok", "output": out, "hops_used": None}, ensure_ascii=False)
    except InputGuardrailTripwireTriggered as e:
        return json.dumps({"status": "input_tripwire", "reason": str(e)}, ensure_ascii=False)
    except OutputGuardrailTripwireTriggered as e:
        return json.dumps({"status": "output_tripwire", "reason": str(e)}, ensure_ascii=False)
    except ToolGuardrailTripwireTriggered as e:
        return json.dumps({"status": "tool_tripwire", "reason": str(e)}, ensure_ascii=False)
    except Exception as e:
        return json.dumps({"status": "error", "reason": f"{type(e).__name__}: {e}"}, ensure_ascii=False)


def reset_runtime_impl() -> str:
    """重置 session + tracer + agents。"""
    global PROD_RT
    PROD_RT = build_prod_runtime()
    return json.dumps({"status": "reset", "session_id": PROD_RT.session.session_id})


def get_trace_impl() -> str:
    """
    Returns:
        json: spans 摘要（避免过度监视：截断 input/output）。
    """
    spans = []
    for s in PROD_RT.tracer.trace.spans[-40:]:
        spans.append(
            {
                "kind": s.kind,
                "name": s.name,
                "error": s.error,
                "ms": None if s.end_ts is None else round((s.end_ts - s.start_ts) * 1000, 1),
                # 刻意不回传完整敏感原文
                "has_input": s.input is not None,
            }
        )
    return json.dumps({"trace_id": PROD_RT.tracer.trace.trace_id, "spans": spans}, ensure_ascii=False)


def get_session_impl() -> str:
    """
    Returns:
        json: 会话历史（截断）。
    """
    msgs = PROD_RT.session.messages[-20:]
    return json.dumps({"session_id": PROD_RT.session.session_id, "messages": msgs}, ensure_ascii=False)


def build_control_tools() -> list[StructuredTool]:
    """
    Returns:
        tools: 运行时控制面。
    """

    def _run(**kwargs: Any) -> str:
        return run_agents_impl(RunArgs(**kwargs).text)

    def _reset(**kwargs: Any) -> str:
        return reset_runtime_impl()

    def _trace(**kwargs: Any) -> str:
        return get_trace_impl()

    def _sess(**kwargs: Any) -> str:
        return get_session_impl()

    return [
        StructuredTool.from_function(
            name="reset_runtime",
            description="Reset agents, session, and tracer.",
            func=_reset,
            args_schema=EmptyArgs,
        ),
        StructuredTool.from_function(
            name="run_agents",
            description="Run triage→(handoff)→billing pipeline with guardrails and tracing.",
            func=_run,
            args_schema=RunArgs,
        ),
        StructuredTool.from_function(
            name="get_trace",
            description="Get truncated span summary for the last runs.",
            func=_trace,
            args_schema=EmptyArgs,
        ),
        StructuredTool.from_function(
            name="get_session",
            description="Get recent session messages.",
            func=_sess,
            args_schema=EmptyArgs,
        ),
    ]


CONTROL_TOOLS = build_control_tools()


def build_control_agent() -> Any:
    """
    Returns:
        agent: 通过工具驱动 OpenAI-Agents 风格运行时。
    """
    system = (
        "You operate an OpenAI-Agents-style runtime.\n"
        "Flow: reset_runtime -> run_agents(text) -> get_trace / get_session.\n"
        "Explain handoffs and guardrail tripwires in Chinese."
    )
    return create_agent(get_llm(), CONTROL_TOOLS, system_prompt=system)


def format_agent_messages(messages: list[BaseMessage]) -> str:
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"ACTION: {tc['name']}({tc.get('args') or {}})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            content = m.content if len(str(m.content)) < 500 else str(m.content)[:500] + "..."
            lines.append(f"OBS[{m.name}]: {content}")
    return "\n".join(lines)


def count_tool_calls(messages: list[BaseMessage]) -> int:
    n = 0
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            n += len(m.tool_calls)
    return n


print(f"OpenAI-Agents-style + LangChain ready | {MODEL}")


## 4. 生产示例：围栏拒绝 + 账单交接 + Trace


In [ ]:
def demo_deepseek_openai_agents() -> None:
    """真实 API；无 key 则 SKIP。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production demo: DEEPSEEK_API_KEY missing")
        return

    print(reset_runtime_impl())

    print("=== input guardrail ===")
    blocked = json.loads(run_agents_impl("pay with 4111111111111111"))
    print(blocked)
    assert blocked.get("status") == "input_tripwire"

    print("=== billing handoff ===")
    print(reset_runtime_impl())
    ok = json.loads(run_agents_impl("我想退款，订单号 88991"))
    print(json.dumps(ok, ensure_ascii=False, indent=2)[:800])
    assert ok.get("status") == "ok"
    assert ok.get("output")

    trace = json.loads(get_trace_impl())
    kinds = {s["kind"] for s in trace.get("spans", [])}
    print("trace kinds:", kinds, "n=", len(trace.get("spans", [])))
    assert "llm" in kinds or "agent" in kinds
    # handoff 视模型是否调用 transfer_to_billing
    sess = json.loads(get_session_impl())
    assert len(sess.get("messages", [])) >= 2

    print("=== control agent ===")
    try:
        agent = build_control_agent()
        result = agent.invoke(
            {
                "messages": [
                    HumanMessage(
                        content=(
                            "reset_runtime，然后 run_agents 问一个账单问题（带订单号），"
                            "再 get_trace，用中文说明是否发生 handoff 以及有哪些 span 类型。"
                        )
                    )
                ]
            }
        )
        print(format_agent_messages(result["messages"]))
        assert count_tool_calls(result["messages"]) >= 2
    except Exception as e:
        print(f"control agent skipped due to LLM error: {type(e).__name__}: {e}")
    print("PRODUCTION DEMO OK")


demo_deepseek_openai_agents()
